# 04 — Frontier-Vergleich (Phase 5)

Eure Hand-Annotation aus Phase 2 (`annotation/meine_gold.csv`) gegen Frontier-LLM-Annotation derselben 12 Anzeigen (`annotation/frontier_gold.csv`). Output: κ-Tabelle, drei Disagreement-Beispiele, Material fuers Make-or-Buy-Memo (`memo_make_or_buy.md` im Repo-Root).

Cheatsheet: `CHEATSHEETS/frontier-llm-workflow.md`.

## Run-Header

| Feld | Wert |
|---|---|
| Datum | _04.06.2026_ |
| Frontier-Modell | _claude-opus-4-6_ |
| Prompt-Variante | derselbe `SYSTEM_PROMPT_B` wie in der 7B-Pipeline |
| Frontier-CSV | `annotation/frontier_gold.csv` |
| Eigene Gold-CSV | `annotation/meine_gold.csv` |

> **Status:** `frontier_gold.csv` enthält echte Frontier-Annotationen (Claude Opus 4.6, 2025-05-16). 
7 Abweichungen gegenüber meine_gold.csv — echter Vergleich möglich.

## Frontier-Daten laden + Schema-Konformität prüfen

In [1]:
import subprocess
import pandas as pd

# Schema-Check frontier_gold.csv
r = subprocess.run(
    ['python', 'annotation/validate.py', 'annotation/frontier_gold.csv'],
    capture_output=True, text=True, cwd='..'
)
print(r.stdout)

<jemalloc>: Unsupported system page size


OK: frontier_gold.csv ist schema-konform.



In [2]:
gold     = pd.read_csv('../annotation/meine_gold.csv').set_index('id')
frontier = pd.read_csv('../annotation/frontier_gold.csv').set_index('id')

print(f'Eigene Gold-CSV:    {len(gold)} Anzeigen')
print(f'Frontier-Gold-CSV:  {len(frontier)} Anzeigen')
common = sorted(set(gold.index) & set(frontier.index))
print(f'Gemeinsame IDs:     {len(common)}')

for field in ['homeoffice', 'vertragsart', 'erfahrungslevel']:
    g = gold.loc[common, field].astype(str)
    f = frontier.loc[common, field].astype(str)
    n = (g == f).sum()
    print(f'{field}: {n}/{len(common)} uebereinstimmend')

Eigene Gold-CSV:    12 Anzeigen
Frontier-Gold-CSV:  12 Anzeigen
Gemeinsame IDs:     12
homeoffice: 11/12 uebereinstimmend
vertragsart: 12/12 uebereinstimmend
erfahrungslevel: 10/12 uebereinstimmend


## κ Mensch ↔ Frontier

**Hypothese (zu prüfen, sobald echte Frontier-Daten vorliegen):** Ein Frontier-Modell mit demselben Schema sollte ein hohes, aber **nicht** perfektes κ erreichen — Disagreements erwarte ich dort, wo das Schema unscharf ist (z.B. `homeoffice` `ja`/`teilweise`, `gehalt_min_eur` bei Näherungsangaben).

In [3]:
# 1) Kopie-Pruefung: ist frontier_gold nur eine Kopie von meine_gold?
import pandas as pd
g_cmp = pd.read_csv('../annotation/meine_gold.csv').set_index('id').sort_index()
f_cmp = pd.read_csv('../annotation/frontier_gold.csv').set_index('id').sort_index()
fields = ['homeoffice','vertragsart','erfahrungslevel','gehalt_min_eur','gehalt_zeitraum','skills_top3']
n_diff = int((g_cmp[fields].astype(str) != f_cmp[fields].astype(str)).sum().sum())
if n_diff == 0:
    print('WARNUNG: frontier_gold.csv ist Zeichen fuer Zeichen identisch mit meine_gold.csv')
    print('-> Das unten berechnete kappa=1.0 ist trivial, KEIN echter Frontier-Vergleich.')
    print('-> Echte Frontier-Annotation noch durchfuehren (siehe Anleitung in der naechsten Markdown-Zelle).')
else:
    print(f'frontier_gold weicht in {n_diff} Zellen von meine_gold ab -> echter Vergleich.')

print()
# 2) Cohen's kappa (Mensch <-> Frontier)
r = subprocess.run(
    ['python', 'annotation/validate.py', 'annotation/meine_gold.csv',
     '--kappa-against', 'annotation/frontier_gold.csv'],
    capture_output=True, text=True, cwd='..'
)
print(r.stdout)

frontier_gold weicht in 7 Zellen von meine_gold ab -> echter Vergleich.

OK: meine_gold.csv ist schema-konform.

Cohen's kappa: meine_gold.csv vs. frontier_gold.csv
Gemeinsame IDs: 12

Feld                    kappa   Übereinst.
------------------------------------------
homeoffice              0.886         92%
vertragsart             1.000        100%
erfahrungslevel         0.786         83%

Interpretation (Landis & Koch 1977):
  < 0.00      schlechter als Zufall
  0.00–0.20   schlecht
  0.21–0.40   mäßig
  0.41–0.60   moderat
  0.61–0.80   substanziell
  0.81–1.00   fast perfekt



### κ-Tabelle (Mensch ↔ Frontier, n = 12)

Die κ-Werte kommen aus der Zelle oben (`validate.py --kappa-against`), nicht aus handgeschriebenen Zahlen.

Niedrigstes κ bei erfahrungslevel (0,786) – also dort die meiste Uneinigkeit, was zur Schema-Unschärfe nicht_genannt vs. junior bei Studierenden/Praktika passt. vertragsart perfekt (κ = 1,0). Insgesamt hohe, aber nicht perfekte Übereinstimmung – ein Frontier-Modell trifft die Schema-Interpretation überwiegend, aber nicht vollständig.

## Drei Disagreement-Beispiele

In [4]:
# Disagreements bei gehalt_min_eur
g_gehalt = gold.loc[common, 'gehalt_min_eur'].fillna(-1).astype(float)
f_gehalt = frontier.loc[common, 'gehalt_min_eur'].fillna(-1).astype(float)

print('=== gehalt_min_eur: Disagreements ===')
gefunden = False
for rid in common:
    if g_gehalt[rid] != f_gehalt[rid]:
        print(f'  {rid}: ich={g_gehalt[rid]}  frontier={f_gehalt[rid]}')
        gefunden = True
if not gefunden:
    print('  Keine Disagreements.')

# Disagreements bei skills_top3 (Set-Match)
print('\n=== skills_top3: Disagreements (Set-Match) ===')
gefunden = False
for rid in common:
    g_set = set(s.strip().lower() for s in str(gold.loc[rid, 'skills_top3'] or '').split('|') if s.strip())
    f_set = set(s.strip().lower() for s in str(frontier.loc[rid, 'skills_top3'] or '').split('|') if s.strip())
    if g_set != f_set:
        print(f'  {rid}: ich={sorted(g_set)}  frontier={sorted(f_set)}')
        gefunden = True
if not gefunden:
    print('  Keine Disagreements.')

=== gehalt_min_eur: Disagreements ===
  Keine Disagreements.

=== skills_top3: Disagreements (Set-Match) ===
  10000-1003-S: ich=['aws', 'python', 'tensorflow']  frontier=['python', 'scikit-learn', 'tensorflow']
  10000-1004-S: ich=['nan']  frontier=['pandas', 'python', 'tableau']
  10000-1009-S: ich=['nan']  frontier=['python', 'r', 'sql']
  10000-1012-S: ich=['nan']  frontier=['power_bi', 'python', 'sql']


## Drei Disagreement-Beispiele

### Fall 1 — 1009 skills_top3 (Frontier deckt eine Gold-Lücke auf). 

- Mein Gold: leer. Frontier: python|r|sql. Die Anzeige nennt die Tools explizit („Stack: Python, R, SQL").
- Wer hatte recht: Frontier hatte recht. Die Tools stehen wörtlich in der Anzeige — mein Gold war eine Lücke, kein bewusster Entscheid. Ich hätte sie eintragen müssen.

### Fall 2 — 1006 homeoffice (Frontier schema-treuer als mein Gold). 

- Mein Gold: teilweise. Frontier: ja. Die Anzeige sagt „Hybrides Arbeiten nach Absprache" – ohne Verhältnis/Wochentage.
- Wer hatte recht: Frontier hatte recht, weil mein Gold der eigenen Regel widerspricht. Nach meiner Definition aus Iteration A gilt: teilweise nur mit explizitem Verhältnis oder Wochentagen. „Nach Absprache" nennt kein Verhältnis — also wäre ja korrekt gewesen.

### Fall 3 — 1004 erfahrungslevel (echte Auslegungssache). 

- Mein Gold: nicht_genannt. Frontier: junior. Werkstudent, „mind. 3. Semester", kein Level genannt. Beide Lesarten vertretbar (kein Level explizit vs. Studi = Einsteiger).
- Wer hatte recht: Beide Lesarten sind vertretbar. „mind. 3. Semester" deutet auf Einsteiger hin, aber kein Erfahrungslevel ist explizit genannt. Ich habe vorsichtiger annotiert, Frontier hat eine Zuordnung gewagt.
- Muster insgesamt: Frontier macht weniger Flüchtigkeitsfehler bei explizit genannten Infos (Skills, Tools). Der Mensch ist bei unscharfen Kategorien vorsichtiger und neigt zu nicht_genannt, wo das Frontier eine Zuordnung wagt.

## Einige Einträge aus Korpus

In [5]:
import json, csv
GOLD_IDS = [f'10000-10{n:02d}-S' for n in range(1, 13)]
corpus = {json.loads(l)['refnr']: json.loads(l)
          for l in open('../daten/eigener_korpus.jsonl') if l.strip()}

block = "\n\n".join(f"=== id: {rid} ===\n{corpus[rid]['text']}" for rid in GOLD_IDS)
print(block)

=== id: 10000-1001-S ===
Die Stadtwerke Bremen AG bietet zum 1. September eine Ausbildung zum/zur Fachinformatiker/in Fachrichtung Daten- und Prozessanalyse an. Während der dreijährigen Ausbildung lernst du, Daten zu erheben, aufzubereiten und auszuwerten. Du arbeitest mit Python, SQL und Excel. Außerdem lernst du, Geschäftsprozesse zu modellieren und zu optimieren. Voraussetzung: mittlerer Schulabschluss oder Abitur. Ausbildungsvergütung nach Tarif (1. Jahr: 1.050 EUR, 2. Jahr: 1.100 EUR, 3. Jahr: 1.200 EUR). Arbeitsort: Bremen, Präsenz erforderlich.

=== id: 10000-1002-S ===
Wir suchen zur Verstärkung unseres Teams einen Data Analyst (m/w/d) in Hamburg. Sie analysieren Bestands- und Transportdaten, erstellen regelmäßige Reports in Power BI und unterstützen das Management mit datengetriebenen Entscheidungsgrundlagen. Anforderungen: abgeschlossenes Studium oder vergleichbare Qualifikation, mind. 2 Jahre Berufserfahrung, sicherer Umgang mit SQL und Python, Power BI-Kenntnisse von Vortei